# Phase 8a.10 — Spout-behavior visual gate

Driver notebook for sub-deliverable A: render the 11 canonical per-mouse plot families plus the cohort-summary tile grids.

**Per CLAUDE.md**: this notebook orchestrates only. All analysis + plotting logic lives in `stroke_orofacial.spout_behavior.visual_gate` (CG-1 through CG-4) and `stroke_orofacial.spout_behavior.cohort_summary` (CG-5).

**Cohort A** (Task 0.5 extension audit): 341 (animal, date) pairs across PS46/47/48/49/50/51/54/55, defined by legacy `basic_behavior_analysis3/<mouse>/report.json[latency_bySession]` keys. Each renderer iterates the bare-date subset for longitudinal plots (b-half sessions excluded; PS48/2025-03-14 merged-NPZ included).

**Outputs**:
- Per-mouse: `pipeline_outputs/spout_behavior/visual_gate/<animal>/<family>{_relStroke|_absDate}.{svg,png}`
- Cohort summary: `pipeline_outputs/spout_behavior/summary_<animals-joined>/<family>{...}.{svg,png}`

**Wall-time note**: a full 8-animal run iterates ~341 sessions across 4 distinct cohort passes (longitudinal-mean families share one pass via `compute_per_session_stats`; lick raster + lickiness + switchiness pass independently). On the MICROSCOPE share this runs ~5-15 min per animal; expect ~45-120 min for the full cohort. The cohort-summary tiler is fast (<1 min) once the per-mouse PNGs are on disk.

In [ ]:
# ============================================================
# USER INPUTS — edit variables below, then run all cells.
# ============================================================

# Cohort A animals. Default = all 8 (Task 0.5 extension audit).
ANIMALS = ["PS46", "PS47", "PS48", "PS49", "PS50", "PS51", "PS54", "PS55"]

# Overwrite policy for per-mouse + cohort-summary outputs.
#   "skip"  — skip targets that already exist (idempotent re-runs)
#   "force" — always overwrite
#   "raise" — raise FileExistsError if any target exists
OVERWRITE = "skip"

# Toggles
INCLUDE_PER_MOUSE_RENDERS = True   # the 11 canonical per-mouse renderers
INCLUDE_COHORT_SUMMARY = True      # cohort-tile grids (CG-5)

# Plot families. None → all 11 canonical families (CG-2 through CG-4).
# Override with an explicit list to render a subset, e.g.:
#   PLOT_FAMILIES = ["lickiness_bySession", "switchiness_bySession"]
PLOT_FAMILIES = None

# Stroke-relative x-axis (True) vs absolute-date x-axis (False) for
# longitudinal plots. Legacy default is True.
ALIGN_TO_STROKE = True

In [ ]:
import logging

from stroke_orofacial.animals import get_animal
from stroke_orofacial.paths import PathResolver
from stroke_orofacial.spout_behavior.cohort_summary import make_summary_figures
from stroke_orofacial.spout_behavior.visual_gate import (
    _compute_per_session_stats,
    load_behavior_report_params,
    load_visual_gate_config,
    render_accuracy_firstCorrectLick,
    render_accuracy_firstCorrectLickAll,
    render_criteria_performance,
    render_latency_bySession,
    render_lick_raster,
    render_lickRate_bySession,
    render_lickiness_bySession,
    render_numLicks_bySession_all,
    render_numLicks_bySession_correct,
    render_numLicks_bySession_correct_and_incorrect,
    render_switchiness_bySession,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

resolver = PathResolver()
config = load_visual_gate_config()
behavior_params = load_behavior_report_params()

# Plot family → render function mapping. Order = display order in the
# per-animal status table below.
ALL_FAMILIES = {
    "criteria_performance":                       render_criteria_performance,
    "accuracy_firstCorrectLick":                  render_accuracy_firstCorrectLick,
    "accuracy_firstCorrectLickAll":               render_accuracy_firstCorrectLickAll,
    "lickRate_bySession":                         render_lickRate_bySession,
    "latency_bySession":                          render_latency_bySession,
    "numLicks_bySession_correct":                 render_numLicks_bySession_correct,
    "numLicks_bySession_all":                     render_numLicks_bySession_all,
    "numLicks_bySession_correct_and_incorrect":   render_numLicks_bySession_correct_and_incorrect,
    "lick_raster":                                render_lick_raster,
    "lickiness_bySession":                        render_lickiness_bySession,
    "switchiness_bySession":                      render_switchiness_bySession,
}

# Families that share the `_compute_per_session_stats` aggregation
# cache (the 8 CG-2 longitudinal-mean families). lick_raster +
# lickiness_bySession + switchiness_bySession iterate Cohort A
# independently — they each consume different per-trial state and
# the _PerSessionStats dataclass doesn't carry it.
LONGITUDINAL_MEAN_FAMILIES = {
    "criteria_performance",
    "accuracy_firstCorrectLick",
    "accuracy_firstCorrectLickAll",
    "lickRate_bySession",
    "latency_bySession",
    "numLicks_bySession_correct",
    "numLicks_bySession_all",
    "numLicks_bySession_correct_and_incorrect",
}

families = list(ALL_FAMILIES.keys()) if PLOT_FAMILIES is None else list(PLOT_FAMILIES)
print(f"Rendering {len(families)} plot families for {len(ANIMALS)} animals.")
print(f"  families: {families}")
print(f"  animals : {ANIMALS}")
print(f"  overwrite: {OVERWRITE}; align_to_stroke: {ALIGN_TO_STROKE}")

In [ ]:
# ============================================================
# Per-mouse loop: compute shared per-session stats once per
# animal, then call each render_* function. The 8 longitudinal-
# mean families consume the shared stats via `stats=` kwarg; the
# other 3 (lick_raster, lickiness, switchiness) iterate Cohort A
# independently.
#
# Per-(animal, family) error handling: log + continue. Status
# table at the end records per-cell outcome.
# ============================================================

status: dict[str, dict[str, str]] = {animal: {} for animal in ANIMALS}

if INCLUDE_PER_MOUSE_RENDERS:
    for animal in ANIMALS:
        print(f"\n=== {animal} ===")
        try:
            animal_info = get_animal(animal)
        except KeyError as exc:
            status[animal]["_animal_info"] = f"ERROR: {exc}"
            print(f"  animal_info: ERROR — {exc}")
            continue

        # Shared stats compute for longitudinal-mean families.
        stats = None
        if any(f in LONGITUDINAL_MEAN_FAMILIES for f in families):
            try:
                stats = _compute_per_session_stats(animal, resolver, behavior_params)
                print(f"  per-session stats: computed for {len(stats.dates)} sessions")
            except Exception as exc:
                logging.warning("  %s: stats compute failed: %s", animal, exc)
                status[animal]["_stats"] = f"ERROR: {type(exc).__name__}: {exc}"
                print(f"  stats: ERROR — {type(exc).__name__}: {exc}")
                # Don't abort the animal; standalone renderers (lick raster,
                # lickiness, switchiness) might still succeed.

        for family in families:
            render_fn = ALL_FAMILIES.get(family)
            if render_fn is None:
                status[animal][family] = "unknown family"
                print(f"  {family}: unknown family (typo?)")
                continue
            kwargs = {
                "align_to_stroke": ALIGN_TO_STROKE,
                "config": config,
                "behavior_params": behavior_params,
                "animal_info": animal_info,
                "overwrite": OVERWRITE,
            }
            if family in LONGITUDINAL_MEAN_FAMILIES and stats is not None:
                kwargs["stats"] = stats
            try:
                paths = render_fn(animal, resolver, **kwargs)
                status[animal][family] = f"ok ({paths[0].name})"
                print(f"  {family}: ok → {paths[0].name}")
            except Exception as exc:
                logging.warning("  %s/%s: %s", animal, family, exc)
                status[animal][family] = f"ERROR: {type(exc).__name__}: {exc}"
                print(f"  {family}: ERROR — {type(exc).__name__}")

In [ ]:
# ============================================================
# Cohort summary tile grids (CG-5). Reads the per-mouse PNGs
# written above; emits one tiled SVG + PNG per non-skipped plot
# family at
# pipeline_outputs/spout_behavior/summary_<animals-joined>/.
# Per visual_gate.summary_skip_plots (defaults.yaml), lick_raster
# is auto-skipped from the cohort tiling.
# ============================================================

summary_status = None
if INCLUDE_COHORT_SUMMARY:
    print("\n=== cohort-summary tiler ===")
    try:
        out_dir = make_summary_figures(
            ANIMALS, resolver,
            plot_families=PLOT_FAMILIES,
            align_to_stroke=ALIGN_TO_STROKE,
            config=config,
            overwrite=OVERWRITE,
        )
        summary_status = f"ok ({out_dir})"
        print(f"  summary saved to: {out_dir}")
    except Exception as exc:
        logging.error("cohort summary failed: %s", exc)
        summary_status = f"ERROR: {type(exc).__name__}: {exc}"
        print(f"  ERROR: {type(exc).__name__}: {exc}")

In [ ]:
# ============================================================
# Per-(animal, family) status table. ‘ok’ = file emitted; ERR =
# log message captured. Look for the per-cell errors first when
# triaging a re-run.
# ============================================================

print("=" * 60)
print("Render status (per animal × family)")
print("=" * 60)
n_ok = n_err = n_unknown = 0
for animal in ANIMALS:
    print(f"\n{animal}:")
    for family, st in status[animal].items():
        if st.startswith("ok"):
            marker = "OK "
            n_ok += 1
        elif st.startswith("ERROR"):
            marker = "ERR"
            n_err += 1
        else:
            marker = "?? "
            n_unknown += 1
        print(f"  [{marker}] {family}: {st}")

print("\n" + "=" * 60)
print(f"Totals: {n_ok} ok, {n_err} errors, {n_unknown} other")
if INCLUDE_COHORT_SUMMARY:
    print(f"Cohort summary: {summary_status}")
print("=" * 60)